# Data Onboarding and Contracts

> **The story:** In 2005, Amazon engineers published the Dynamo paper after learning that availability depends on explicit identity, versioning, and failure behavior. Retrieval systems inherit the same lesson: a document is not just text; it is text plus authority, lifecycle, and provenance. Riverside House now needs that discipline across policy PDFs, manuscripts, ERP exports, and APIs.
>
> **Where you are:** The FDE route has bounded the Riverside engagement and selected authorized retrieval as part of the smallest credible architecture. But the data gate is still open: 6 source systems claim useful evidence, while 14 frozen samples contain stale policy, parser, schema, deletion, and access failures. This notebook produces `DATA-01` through `DATA-05`; the next chapter consumes the unresolved identity and ACL gaps.
>
> **Notation:** $s$ is a source; $r$ is a source record; $d$ is stable document identity; $v$ is a source version; $h(r)$ is a content hash; $A(r)$ is the ACL; $L(r)$ is lifecycle state; $w$ is a durable watermark; $Q$ is quarantine; and $I$ is the current retrieval index.

> **Inter-notebook contract:** This chapter reads frozen fixture version `RIV-FDE-1.0.0` and produces data-readiness artifacts. It writes no production artifacts. The identity chapter must treat unresolved ACL freshness and purpose as open inputs.

**Evidence banner:** `LOCAL FIXTURE`, `SYNTHETIC`, `EXECUTION VERIFIED THEN CLEARED`, `NO CUSTOMER VALIDATION`, `NO DATABRICKS VALIDATION`.

## 0 - The Challenge

> **The mission:** Riverside House - make current, authorized editorial evidence retrievable without indexing stale, malformed, over-broad, or undeletable records.

**What we know so far:**
- The frozen case names 6 sources across PDF, text, ERP, and API shapes.
- The source fixture contains 14 deterministic records.
- The platform contract requires identity, source version, content hash, ACL, region, classification, ingestion time, pipeline version, and deletion state.
- **But a payload-presence check would accept all 14 records and preserve every seeded failure.**

**What's blocking us:** A superseded policy can rank as current; PDF layout can change meaning; duplicate pages can crowd retrieval; an ERP null can become worldwide rights; a renamed API field can become a silent null; a deleted autosave can survive in the index; and a disabled contractor can remain authorized through stale groups.

**What this chapter unlocks:** A source-by-source decision backed by deterministic fixture checks, explicit quarantine and tombstone paths, fail-closed ACL logic, complete lineage, and an overall retrieval-readiness verdict. Route validation executed these checks successfully against the committed synthetic fixtures and then cleared the generated outputs. The run is local fixture evidence only, not customer or production data readiness.

```mermaid
flowchart LR
    A["6 sources / 14 records"] --> B["Naive flatten and upsert"]
    B --> C["Failure: stale, malformed, over-broad"]
    C --> D["Contracts and quarantine"]
    D --> E["Lifecycle and ACL gates"]
    E --> F["DATA-05 readiness verdict"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Topic | Coverage | Why |
|---|---|---|
| Inventory, mapping, quarantine, versions, drift, ACLs, lineage, deletion, sync, verdict | Built and validated against synthetic fixtures | The cleared route run verifies local logic; retain a separate run record before citing observations |
| Delta tables, Spark jobs, vector index updates | Explained and linked | Owned by the RAG Knowledge Pipeline |
| OCR benchmark and customer retention approval | Named only | Require representative binaries and authorized owners |

**Output contract:** The route validation output was cleared. `templates/quality-report.json` and `templates/notebook-output-record.md` remain `not_run` until a learner or practicum execution retains environment, source commit, fixture version, method, result, and limitations.

In [ ]:
# -- Load and validate the frozen case ---------------------------------------
from collections import Counter, defaultdict
from copy import deepcopy
from pathlib import Path
import json

from jsonschema import Draft202012Validator

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "AUTHORING_GUIDE.md").is_file() and (candidate / "learning" / "fde" / "shared").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the ai-portfolio repository.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
SHARED_DIR = REPO_ROOT / "learning" / "fde" / "shared"
FIXTURE_DIR = SHARED_DIR / "fixtures"
SCHEMA_DIR = SHARED_DIR / "schemas"

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

engagement = load_json(FIXTURE_DIR / "riverside-engagement-v1.json")
samples = load_json(FIXTURE_DIR / "riverside-source-samples-v1.json")
facts = load_json(FIXTURE_DIR / "expected-facts-v1.json")
for name, value, schema_name in (
    ("engagement", engagement, "riverside-engagement.schema.json"),
    ("samples", samples, "riverside-source-samples.schema.json"),
    ("facts", facts, "expected-facts.schema.json"),
):
    errors = sorted(
        Draft202012Validator(load_json(SCHEMA_DIR / schema_name)).iter_errors(value),
        key=lambda error: list(error.path),
    )
    assert not errors, f"{name}: {errors[0].message if errors else 'schema error'}"

records = samples["records"]
print(f"Case: {engagement['fixture_id']} | version: {engagement['fixture_version']}")
print(f"Sources: {len(engagement['source_inventory'])} | sample records: {len(records)}")
print(f"Expected facts: {len(facts['facts'])}")

**Predict:** If a record is ready whenever it has a `document_id` and payload, do you get A) all 14, B) fewer than 14 because controls participate, or C) zero because fixtures cannot teach a mechanism? Choose before the next cell.

In [ ]:
# -- Expose the naive readiness failure --------------------------------------
naive_ready = [record for record in records if record.get('document_id') and record.get('payload')]
non_index_decisions = {
    'exclude_from_current_policy_index', 'quarantine_until_duplicate_and_ocr_review',
    'exclude_and_emit_tombstone', 'quarantine_for_owner_resolution',
    'quarantine_schema_drift', 'preserve_for_incident_replay',
    'allow_with_request_purpose', 'deny_and_reconcile',
}
controlled_candidates = [record for record in records if record['expected_onboarding']['decision'] not in non_index_decisions]
print(f'[Measured - local fixture] Naive ready count: {len(naive_ready)} of {len(records)}')
print(f'[Measured - local fixture] Candidates before deeper gates: {len(controlled_candidates)}')
print('Prediction resolution: A is the naive result; B is the required control model.')

## 1 - Source Inventory Is an Authority Contract

An inventory row matters only when it names who can authorize purpose, retention, deletion, and access. Riverside's counts and freshness targets remain `customer_claim` evidence. Counting rows locally does not validate those claims.

```mermaid
flowchart LR
    A["Frozen inventory"] --> B["Join by source_id"]
    B --> C{"Owner, purpose, ACL, refresh, deletion known?"}
    C -->|No| D["Block and assign owner"]
    C -->|Yes| E["Sample approved scope"]
    E --> F["DATA-01"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Treat `estimated_records` as observed | A planning claim becomes fake measurement |
| Right | Preserve `customer_claim` and assign a validation owner | Sampling can proceed without inventing truth |
| Wrong | Join by position or display name | Reordering or aliases bind the wrong authority |
| Right | Join immutable IDs | Identity survives naming changes |

**Your turn:** Change `required_samples_per_source` from `1` to `3`. Which sources become blocked, and is that threshold measured or invented?

**Quick Health Check:** fixture versions match; every record joins one source; source types and owners are explicit; claims retain their class.

In [ ]:
# -- Build and check DATA-01 -------------------------------------------------
inventory = {source['source_id']: source for source in engagement['source_inventory']}
records_by_source = defaultdict(list)
for record in records:
    records_by_source[record['source_id']].append(record)
orphan_ids = sorted(record['record_id'] for record in records if record['source_id'] not in inventory)
unsampled_ids = sorted(set(inventory) - set(records_by_source))
source_types = sorted({source['source_type'] for source in inventory.values()})
required_samples_per_source = 1  # CHANGE THIS: try 3, then justify it.
sample_shortfalls = {source_id: required_samples_per_source - len(records_by_source[source_id]) for source_id in inventory if len(records_by_source[source_id]) < required_samples_per_source}
assert samples['fixture_version'] == engagement['fixture_version'] == 'RIV-FDE-1.0.0'
assert not orphan_ids and not unsampled_ids
assert set(source_types) == {'API', 'ERP', 'PDF', 'text'}
assert all(source['owner_person_id'] for source in inventory.values())
print(f'[Measured - local fixture] Sources joined: {len(inventory)}; types: {source_types}')
print(f'[Measured - local fixture] Orphans: {orphan_ids}; unsampled: {unsampled_ids}')
print(f'Policy result at sample threshold {required_samples_per_source}: {sample_shortfalls}')
print('[Customer claim retained] Counts, freshness, ownership, and deletion behavior remain unvalidated.')

## 2 - Mapping, Parsing, and Quarantine

Flattening every payload destroys heading relationships, hides OCR uncertainty, turns null territory into an invitation to guess, and converts renamed required fields into nulls. A mapping contract decides what survives and where uncertainty goes.

```mermaid
flowchart LR
    A["PDF, text, ERP, API"] --> B["Typed adapter"]
    B --> C{"Contract and quality pass?"}
    C -->|No| D["Quarantine with safe reason"]
    C -->|Deleted| E["Tombstone"]
    C -->|Yes| F["Parsed document v1"]
    D --> G["Owner review and replay"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Which shortcut is safe: flatten PDF blocks, default missing territory to worldwide, or accept a renamed API field? None is safe; the next cell measures each boundary.

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | `territory or 'worldwide'` | Missing evidence becomes an unsupported grant |
| Right | Quarantine and assign the rights owner | Uncertainty stays non-retrievable |
| Wrong | Log raw text in quarantine | Sensitive content leaks through operations |
| Right | Safe error code plus lineage | Reviewers replay from governed storage |

**Quick Health Check:** stale version, layout, duplicate page, low OCR, tombstone, null territory, and required-field drift each reach a distinct path.

In [ ]:
# -- Build DATA-02 mappings and DATA-03 quality evidence ---------------------
def repeated_pdf_pages(record: dict) -> list[int]:
    if record['payload_type'] != 'pdf_pages':
        return []
    seen, duplicates = set(), []
    for page in record['payload']['pages']:
        fingerprint = json.dumps(page.get('text_blocks', []), sort_keys=True)
        if fingerprint in seen:
            duplicates.append(page['page_number'])
        seen.add(fingerprint)
    return duplicates

def local_disposition(record: dict, ocr_threshold: float = 0.80) -> tuple[str, list[str]]:
    reasons, payload = [], record['payload']
    if record['lifecycle']['status'] == 'superseded': reasons.append('not_current')
    if 'deleted' in record['deletion_state']: reasons.append('tombstone_required')
    if record['payload_type'] == 'pdf_columns': reasons.append('layout_parser_required')
    duplicates = repeated_pdf_pages(record)
    if duplicates: reasons.append(f'duplicate_pages:{duplicates}')
    if record['payload_type'] == 'pdf_pages':
        low_ocr = [page['page_number'] for page in payload['pages'] if page.get('ocr_confidence', 1.0) < ocr_threshold]
        if low_ocr: reasons.append(f'low_ocr_pages:{low_ocr}')
    if record['payload_type'] == 'erp_row' and 'territory' in payload and payload['territory'] is None: reasons.append('territory_unknown')
    if record['payload_type'] == 'api_page' and any('status' not in item for item in payload['items']): reasons.append('required_status_schema_drift')
    if any(reason.startswith(('duplicate_pages', 'low_ocr_pages')) for reason in reasons): return 'QUARANTINE_PARSE', reasons
    if 'territory_unknown' in reasons or 'required_status_schema_drift' in reasons: return 'QUARANTINE_CONTRACT', reasons
    if 'tombstone_required' in reasons: return 'TOMBSTONE', reasons
    if 'not_current' in reasons: return 'EXCLUDE_VERSION', reasons
    if 'layout_parser_required' in reasons: return 'CONDITIONAL_PARSE', reasons
    return 'ACCEPT_MAPPING', reasons

mapping_results = {record['record_id']: local_disposition(record) for record in records}
disposition_counts = Counter(result[0] for result in mapping_results.values())
quality_report = {
    'artifact_id': 'DATA-03',
    'claim_class': 'Measured - local fixture',
    'fixture_version': samples['fixture_version'],
    'sample_size': len(records),
    'disposition_counts': dict(sorted(disposition_counts.items())),
    'quarantine_record_ids': sorted(
        record_id for record_id, result in mapping_results.items()
        if result[0] in {'QUARANTINE_PARSE', 'QUARANTINE_CONTRACT'}
    ),
    'limitations': [
        'Synthetic seeded-shape checks are not representative parser benchmarks.',
        'No customer source, Databricks job, or vector index was tested.',
    ],
}
assert mapping_results['REC-RIV-PDF-001'][0] == 'EXCLUDE_VERSION'
assert mapping_results['REC-RIV-PDF-003'][0] == 'CONDITIONAL_PARSE'
assert mapping_results['REC-RIV-PDF-004'][0] == 'QUARANTINE_PARSE'
assert mapping_results['REC-RIV-TEXT-002'][0] == 'TOMBSTONE'
assert mapping_results['REC-RIV-ERP-002'][0] == 'QUARANTINE_CONTRACT'
assert mapping_results['REC-RIV-API-002'][0] == 'QUARANTINE_CONTRACT'
print(f"[Measured - local fixture] DATA-03 sample size: {quality_report['sample_size']}")
print(f"[Measured - local fixture] Dispositions: {quality_report['disposition_counts']}")
print(f"[Measured - local fixture] Quarantine: {quality_report['quarantine_record_ids']}")
print('Prediction resolution: every shortcut violates a different contract boundary.')
print('Limitation: seeded-shape detection is not a parser benchmark.')

### Code Walkthrough: `local_disposition`

1. **Lifecycle precedes parse quality** - deleted and superseded records are not rescued by clean text.
2. **Layout carries meaning** - column association is part of the mapping contract.
3. **Duplicate and OCR checks stay separate** - removing repetition does not make unreadable text trustworthy.
4. **Null is not a business default** - missing territory enters quarantine, never worldwide rights.
5. **Required-field renames fail closed** - an alias needs a versioned mapping decision.

The production equivalent exists in the [remote ingestion pipeline](../../../projects/rag-knowledge-pipeline/phase1-ingest/src/remote/pipeline.py). This notebook keeps the mechanism visible; it does not copy Spark, Delta, or Databricks code.

## 3 - Deduplication and Versioning Without Identity Loss

Similarity says whether bodies look alike. It does not say whether they share authority, lifecycle, or identity. The deleted chapter 38 autosave resembles the current chapter; collapsing them can resurrect deleted text or discard the current version.

```mermaid
flowchart LR
    A["tenant + canonical source URI"] --> B["Stable document identity"]
    B --> C["source_version + content_hash"]
    C --> D{"Lifecycle current?"}
    D -->|No| E["Retain lineage; exclude or tombstone"]
    D -->|Yes| F["Current document view"]
    F --> G["Versioned chunks and index"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Deduplicate globally on content hash | Boilerplate crosses tenant, ACL, and lifecycle |
| Right | Scope identity and versions first | Detection cannot merge authority |
| Wrong | Overwrite old rows | Earlier parser and index decisions vanish |
| Right | Immutable versions plus current view | Changes remain reversible |

**Your turn:** Set `include_superseded = True`. The assertion should fail because the 2024 policy returns.

**Quick Health Check:** old policy excluded, current policy present, deleted autosave absent, stable version keys unique.

In [ ]:
# -- Measure and check version selection ------------------------------------
def stable_version_key(record: dict) -> tuple[str, str, str, str]:
    return record['tenant_id'], record['document_id'], record['source_version'], record['content_hash']
naive_policy_ids = [record['document_id'] for record in records if record['source_id'] == 'SRC-PDF-POLICY-001']
include_superseded = False  # CHANGE THIS: True should fail the current-view check.
selected_policy_ids = [record['document_id'] for record in records if record['source_id'] == 'SRC-PDF-POLICY-001' and (include_superseded or record['lifecycle']['status'] == 'current') and record['deletion_state'] == 'active']
current_ids = [record['document_id'] for record in records if record['lifecycle']['status'] == 'current' and record['deletion_state'] == 'active']
version_keys = [stable_version_key(record) for record in records]
assert 'DOC-POL-AI-2024' in naive_policy_ids and 'DOC-POL-AI-2024' not in selected_policy_ids
assert 'DOC-POL-AI-2026' in selected_policy_ids
assert 'DOC-MANUSCRIPT-ARIA-038-AUTOSAVE' not in current_ids
assert len(version_keys) == len(set(version_keys))
print(f'[Measured - local fixture] Naive policies: {naive_policy_ids}')
print(f'[Measured - local fixture] Current policies: {selected_policy_ids}')
print('PASS: lifecycle selection excludes stale and deleted records without erasing lineage.')

## 4 - Schema Drift, Pagination, and Incremental Sync

Page one reports 100 workflow records and a cursor. Page two adds 37 but renames `status` to `workflow_status` without a contract version. One call loses 27 percent of the stated population; permissive parsing writes null status for what it later finds.

```mermaid
flowchart LR
    A["Page 1: 100"] --> B{"next_cursor?"}
    B -->|Yes| C["Page 2: 37"]
    C --> D{"Required schema exact?"}
    D -->|No| E["Quarantine; keep watermark"]
    D -->|Yes| F["Idempotent merge"]
    F --> G["Commit watermark"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Does traversal reveal 100 or 137 records, and does page two become acceptable?

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Save cursor before durable gates finish | Retry skips records |
| Right | Commit after idempotent accepted writes | Replay stays safe |
| Wrong | Treat timeout as proof write failed | Retry duplicates side effect |
| Right | Business idempotency key plus reconciliation | Response loss is recoverable |

**Your turn:** Set `accept_undocumented_alias = True`. A value appears, but the v1 contract check must fail.

**Quick Health Check:** follow cursors, compare exact fields, quarantine drift, retain overlap, reconcile, and commit state after acceptance.

In [ ]:
# -- Measure and check pagination, drift, and idempotency -------------------
workflow_pages = sorted([record for record in records if record['payload_type'] == 'api_page'], key=lambda record: record['payload']['page'])
one_call_count = workflow_pages[0]['payload']['records_returned']
cursor_count = sum(record['payload']['records_returned'] for record in workflow_pages)
required_fields = {'task_id', 'title_id', 'status', 'assigned_user_id', 'updated_at'}
drift = []
for record in workflow_pages:
    for item in record['payload']['items']:
        missing = sorted(required_fields - set(item))
        if missing: drift.append({'record_id': record['record_id'], 'missing': missing, 'unexpected': sorted(set(item) - required_fields)})
accept_undocumented_alias = False  # CHANGE THIS only with versioned approval.
page_two_item = workflow_pages[1]['payload']['items'][0]
status = page_two_item.get('status')
if accept_undocumented_alias: status = status or page_two_item.get('workflow_status')
history = next(record for record in records if record['record_id'] == 'REC-RIV-API-003')
assert workflow_pages[0]['payload']['next_cursor'] == 'cursor-page-2'
assert cursor_count == 137 and len(drift) == 1 and status is None
assert all(attempt['idempotency_key'] is None for attempt in history['payload']['attempts'])
print(f'[Measured - local fixture] One call: {one_call_count}; cursor complete: {cursor_count}; truncation: {(cursor_count-one_call_count)/cursor_count:.1%}')
print(f'[Measured - local fixture] Drift: {drift}')
print('Prediction resolution: traversal finds 137; required-field drift blocks page two.')
print('Limitation: this does not prove API replay, cursor expiry, or Delta merge behavior.')

## 5 - ACL Projection Must Fail Closed

Authorization combines current identity state, tenant, region, title assignment, role, and purpose. Riverside's disabled contractor still has a stale `ROLE-EDITOR` group; role-only filtering grants access after the contract ended.

```mermaid
flowchart LR
    A["Request context"] --> B{"Identity enabled?"}
    B -->|No| C["Deny and reconcile"]
    B -->|Yes| D{"Tenant, region, title, ACL, purpose match?"}
    D -->|No| C
    D -->|Yes| E["Authorized candidate"]
    E --> F["Audit without text"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Stale role filtering sees `ROLE-EDITOR`. Will current-context authorization allow the contractor?

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Filter copied roles only | Disabled users remain authorized |
| Right | Intersect current context with record ACL | Every candidate uses current authority |
| Wrong | Tenant membership implies purpose | Identity is reused for unrelated access |
| Right | Require purpose and audit | Access stays tied to workflow |

**Quick Health Check:** allowed editor/title, denied rights schedule, disabled contractor, and wrong-purpose request produce reasons.

In [ ]:
# -- Measure and check ACL decisions ----------------------------------------
identity_records = {record['document_id']: record for record in records if record['payload_type'] == 'identity_record'}
manuscript = next(record for record in records if record['document_id'] == 'DOC-MANUSCRIPT-ARIA-037')
rights = next(record for record in records if record['document_id'] == 'DOC-RIGHTS-ARIA-001')
def acl_role_scope(entry: str) -> tuple[str, str | None]:
    role, separator, scope = entry.partition(':')
    return role, scope if separator else None
def authorize(identity_record: dict, record: dict, purpose: str) -> tuple[bool, str]:
    identity = identity_record['payload']
    if not identity.get('enabled', False): return False, 'identity_disabled'
    if record['tenant_id'] not in identity.get('tenant_ids', []): return False, 'tenant_mismatch'
    if record['region'] != identity.get('region_id'): return False, 'region_mismatch'
    title_id = record['payload'].get('title_id')
    if title_id and title_id not in identity.get('title_ids', []): return False, 'title_mismatch'
    if purpose not in {'editorial_retrieval', 'rights_review'}: return False, 'purpose_not_allowed'
    roles = set(identity.get('role_ids', identity.get('direct_role_ids', [])))
    if any(role in roles and (scope is None or scope == title_id) for role, scope in map(acl_role_scope, record['acl'])): return True, 'acl_match'
    return False, 'acl_no_match'
editor, contractor = identity_records['API-USER-EDITOR-017'], identity_records['API-USER-CONTRACTOR-044']
stale_roles = set(contractor['payload']['stale_nested_group_role_ids'])
naive_allowed = any(acl_role_scope(entry)[0] in stale_roles for entry in manuscript['acl'])
decisions = {
    'editor_manuscript': authorize(editor, manuscript, 'editorial_retrieval'),
    'editor_rights': authorize(editor, rights, 'editorial_retrieval'),
    'contractor_manuscript': authorize(contractor, manuscript, 'editorial_retrieval'),
    'wrong_purpose': authorize(editor, manuscript, 'unrelated_analytics'),
}
assert naive_allowed is True
assert decisions['editor_manuscript'] == (True, 'acl_match')
assert decisions['editor_rights'] == (False, 'acl_no_match')
assert decisions['contractor_manuscript'] == (False, 'identity_disabled')
assert decisions['wrong_purpose'] == (False, 'purpose_not_allowed')
print(f'[Measured - local fixture] Naive contractor allowed: {naive_allowed}')
for name, decision in decisions.items(): print(f'[Measured - local fixture] {name}: {decision}')
print('Prediction resolution: stale roles allow; current identity state denies first.')
print('Limitation: this does not prove IdP or vector-filter enforcement.')

## 6 - Lineage, Deletion, and Reconciliation

A tombstone works only if every derived record can be found. Lineage runs from raw document through parsed document, chunks, vectors, and index version. Deletion is a state transition plus downstream absence evidence, not best-effort file removal.

```mermaid
flowchart LR
    A["Source event or reconciliation"] --> B["Raw version"]
    B --> C["Parsed version"]
    C --> D["Versioned chunks"]
    D --> E["Vector records"]
    A --> F{"Delete or revoke?"}
    F -->|Yes| G["Tombstone and derived deletes"]
    G --> H["Negative query or receipt"]
    F -->|No| I["Advance accepted watermark"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

Use [Databricks Index Update Assets](../../../projects/rag-knowledge-pipeline/databricks/indexing/OPERATIONS.md) for the production delete implementation instead of rebuilding it here.

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Delete source row and assume vectors disappear | Derived records remain queryable |
| Right | Trace keys and collect completion evidence | Deletion has a reviewable end |
| Wrong | Depend on deltas forever | Missed events never repair |
| Right | Overlap and reconcile | Speed does not sacrifice correctness |

**Your turn:** Set `overlap_seconds` to `0`. The assertion fails; the correct value still needs measured source-delay evidence.

**Quick Health Check:** complete lineage, replay-safe keys, overlap, reconciliation, tombstone propagation, current-version survival, and target delete evidence.

In [ ]:
# -- Build and check DATA-04 lineage and deletion ---------------------------
required_lineage = {'tenant_id', 'document_id', 'source_uri', 'source_version', 'content_hash', 'acl', 'region', 'classification', 'ingested_at', 'pipeline_version', 'deletion_state'}
lineage_missing = {record['record_id']: sorted(required_lineage - set(record)) for record in records if required_lineage - set(record)}
lineage_coverage = (len(records) - len(lineage_missing)) / len(records)
initial_index = {record['document_id']: record['record_id'] for record in records if record['payload_type'] == 'text'}
index_after_sync = deepcopy(initial_index)
tombstone_ids = [record['document_id'] for record in records if 'deleted' in record['deletion_state']]
for document_id in tombstone_ids: index_after_sync.pop(document_id, None)
overlap_seconds = 300  # CHANGE THIS: 0 removes late-arrival protection.
reconciliation_enabled = True
assert lineage_coverage == 1.0
assert 'DOC-MANUSCRIPT-ARIA-038-AUTOSAVE' in initial_index and 'DOC-MANUSCRIPT-ARIA-038-AUTOSAVE' not in index_after_sync
assert 'DOC-MANUSCRIPT-ARIA-038' in index_after_sync
assert overlap_seconds > 0 and reconciliation_enabled
print(f'[Measured - local fixture] Lineage coverage: {lineage_coverage:.1%}; tombstones: {tombstone_ids}')
print(f'[Measured - local simulation] Before: {sorted(initial_index)}; after: {sorted(index_after_sync)}')
print('External validation required: target deletion, retention, and reconciliation timing.')

## 7 - Retrieval Readiness Is a Weakest-Gate Decision

An aggregate parse rate can hide one unsafe source behind clean records. Readiness is per source and fail closed. Data readiness only permits retrieval evaluation; it proves neither relevance, citations, nor answer quality.

```mermaid
flowchart LR
    A["DATA-01 inventory"] --> B["DATA-02 mapping"]
    B --> C["DATA-03 quality"]
    C --> D["DATA-04 lifecycle and lineage"]
    D --> E{"Every material gate passes?"}
    E -->|No| F["BLOCKED or EXCLUDED"]
    E -->|Conditions| G["CONDITIONAL"]
    E -->|Yes| H["READY FOR RETRIEVAL EVALUATION"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Will any source be production-ready from this notebook? No. Which sources can proceed toward bounded retrieval evaluation?

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Average source metrics | Rights or ACL blockers disappear |
| Right | Weakest material gate controls exposure | Unsafe sources stay excluded |
| Wrong | Call data readiness `RAG ready` | Retrieval and generation remain untested |
| Right | Say `ready for retrieval evaluation` | The next gate stays visible |

**Quick Health Check:** per-source verdicts, blockers, evidence classes, owners, external validations, expiry triggers, and a separate retrieval gate.

In [ ]:
# -- Build and check DATA-05 retrieval readiness ----------------------------
blocking = {'QUARANTINE_PARSE', 'QUARANTINE_CONTRACT'}
conditional = {'CONDITIONAL_PARSE', 'TOMBSTONE', 'EXCLUDE_VERSION'}
source_verdicts = {}
for source_id, source_records in records_by_source.items():
    dispositions = {mapping_results[record['record_id']][0] for record in source_records}
    if source_id == 'SRC-API-IDENTITY-001':
        verdict, rationale = 'BLOCKED', 'Disabled identity and stale nested group require reconciliation.'
    elif dispositions & blocking:
        verdict, rationale = 'BLOCKED', 'At least one sample is in parser or contract quarantine.'
    elif dispositions & conditional:
        verdict, rationale = 'CONDITIONAL', 'Lifecycle, parser, or deletion controls need owner evidence.'
    else:
        verdict, rationale = 'CONDITIONAL', 'Local mapping passes; authority and target enforcement are unvalidated.'
    source_verdicts[source_id] = {'verdict': verdict, 'rationale': rationale, 'external_validation_required': True}
overall_verdict = 'BLOCKED' if any(result['verdict'] == 'BLOCKED' for result in source_verdicts.values()) else 'CONDITIONAL'
assert overall_verdict == 'BLOCKED'
for source_id in ('SRC-PDF-RIGHTS-001', 'SRC-ERP-CATALOG-001', 'SRC-API-WORKFLOW-001', 'SRC-API-IDENTITY-001'): assert source_verdicts[source_id]['verdict'] == 'BLOCKED'
for source_id in ('SRC-PDF-POLICY-001', 'SRC-TEXT-MANUSCRIPT-001'): assert source_verdicts[source_id]['verdict'] == 'CONDITIONAL'
assert all(result['external_validation_required'] for result in source_verdicts.values())
print(f'[Measured - local fixture] Overall DATA-05 verdict: {overall_verdict}')
for source_id, result in source_verdicts.items(): print(f"  {source_id}: {result['verdict']} - {result['rationale']}")
print('Prediction resolution: no production claim exists; every source is conditional or blocked.')
print('Next gate: retrieval and citation evaluation for approved current views only.')

## 8 - Coverage and Handoff

```mermaid
flowchart LR
    A["Local fixture proof code"] --> B["DATA-01 through DATA-05"]
    B --> C["Authorized local run record"]
    C --> D["Databricks validation"]
    C --> E["Identity validation"]
    D --> F["Retrieval evaluation"]
    E --> F
    F --> G["Customer readiness decision"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Three-tier coverage ledger

| Tier | Techniques | Evidence or reason |
|---|---|---|
| Built with executable proof code | Inventory, mappings, quarantine, versions, pagination, drift, ACL negatives, lineage, tombstone, verdict | Deterministic fixture checks are present; committed outputs remain empty |
| Explained and illustrated | Watermark overlap and reconciliation | Fixture cannot measure delay distributions |
| Explained and linked | Delta merges, chunks, embeddings, index deletes | Existing RAG Knowledge Pipeline |
| Named with a reason | OCR benchmark | Needs representative binaries and ground truth |
| Named with a reason | Retention and legal authority | Needs authorized owners |
| Named with a reason | Cloud RBAC, networking, scale, cost, region | Needs target tests |

If a named technique is absent from this ledger, that is the coverage bug this section exists to catch.

### Roadmap checkpoint

| Constraint | Naive state | Proof-code expectation after an authorized run | Current committed status |
|---|---|---|---|
| Record readiness | Payload presence accepts 14 of 14 records | Every seeded lifecycle, parser, schema, deletion, and stale-access failure reaches a distinct path | Verified synthetic run; outputs cleared |
| Source decision | Aggregate quality can hide a blocked source | Four sources block and two remain conditional under the frozen fixture | Expected from fixture; not stored measurement |
| Deletion | Source-row removal can leave derived vectors | Tombstone, lineage, target delete, and negative-query evidence remain linked | Target behavior externally unvalidated |
| Authorization | Copied stale role can allow a disabled contractor | Current identity status denies before record ACL | Local mechanism only; IdP/index enforcement unvalidated |
| Retrieval readiness | Data quality can be mislabeled as RAG readiness | Data gate permits retrieval evaluation only | Customer and Databricks validation open |

### Key takeaways

1. A document contract includes authority, lifecycle, and lineage; text alone is not the record.
2. Quarantine uncertainty before it becomes a permissive default.
3. Similarity suggests duplicates; identity and lifecycle decide current state.
4. Incremental sync needs overlap, idempotency, tombstones, and reconciliation.
5. Authorization uses current request context, not yesterday's copied ACL.
6. Data readiness permits retrieval evaluation; it proves neither RAG quality nor production readiness.

**Output handoff:** Populate `templates/quality-report.json` and `templates/notebook-output-record.md` only after an authorized run records environment, commit, fixture version, method, observations, and limitations.

**Forward:** Carry identity freshness, purpose, and title-scope gaps into `04-identity-isolation-and-compliance/`. Carry approved current views into [Hybrid Search](../../genai/04-rag/04-hybrid-search.ipynb) and [RAG Evaluation](../../genai/04-rag/05-rag-evaluation.ipynb).